**ATACseq Data with over 500k entries** <p>  
each row (=entry) is a OCR peak. The set tells us how strong the peak is in each of the 89 immune cell types  
The data also includes Metadata for each of those peaks:  

- **chrom**  
Chromosome on which the OCR is located  

- **Summit**  
Genomic coordinate of the "peak summit"  
--> position with the strongest ATACseq signal within the OCR  

- **mm10 cons_score**  
as in "evolutionary conservation score"  
--> functional importance (highest value = 1.00)  

- **-log10_bestPvalue**  
remember: p-value measures likelihood of the result being just background noise (small p-value --> likely not background noise)  
--> kleiner log, großer p-value, wahrscheinlich relevant

- **included in systematic analysis**  
Boolean / yes-no flag indicating whether the authors considered this OCR reliable enough for downstream analyses in the paper  

- **TSS**  
name of closest TSS gene

- **genes within 100kb**  
just lists all the genes in that range as str 


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path
ATAC_path = Path('data') / 'ImmGenATAC18_AllOCRsInfo.csv'
ATAC_data = pd.read_csv(ATAC_path)

df = pd.read_csv("data/ImmGenATAC18_AllOCRsInfo.csv")
df.head()

,ImmGenATAC1219.peakID,chrom,Summit,mm10.60way.phastCons_scores,_-log10_bestPvalue,Included.in.systematic.analysis,TSS,genes.within.100Kb,LTHSC.34-.BM,LTHSC.34+.BM,...,DC.4+.Sp,DC.8+.Sp,DC.pDC.Sp,DC.103+11b+.SI,DC.103+11b-.SI,FRC.SLN,IAP.SLN,BEC.SLN,LEC.SLN,Ep.MEChi.Th
0,ImmGenATAC1219.peak_1,chr1,3020786,0.00,0.56,NaN,NaN,NaN,0.41,0.71,...,0.10,0.10,3.19,1.37,0.52,1.27,0.10,0.57,3.27,1.41
1,ImmGenATAC1219.peak_2,chr1,3087226,0.00,0.50,NaN,NaN,NaN,0.41,1.64,...,1.70,0.10,1.41,0.47,0.11,0.92,0.98,2.16,2.34,0.94
2,ImmGenATAC1219.peak_3,chr1,3120109,0.07,10.80,1.0,NaN,NaN,2.36,0.10,...,0.87,0.54,2.72,0.95,0.11,63.38,8.92,1.33,1.04,0.11
3,ImmGenATAC1219.peak_4,chr1,3121485,0.15,3.02,1.0,NaN,NaN,0.41,0.10,...,0.44,1.83,0.66,0.11,0.92,13.50,0.98,1.28,1.04,0.11
4,ImmGenATAC1219.peak_5,chr1,3372787,0.03,1.31,NaN,NaN,NaN,0.41,0.10,...,0.44,0.10,0.66,1.79,0.51,0.92,0.75,1.33,1.61,4.50


<h1> ATAC_sequence Data_Clean_Up 
<h2> Handling Missing Values

In [5]:
missing_value = ATAC_data.isnull().sum()
ATAC_data.isnull().sum()[missing_value > 0]

Included.in.systematic.analysis    177716
TSS                                498303
genes.within.100Kb                  84885
dtype: int64

**Missing value handling**: <p>
Wir haben nur in drei Kategorien missing values (NAs). In allen drei Kategorien steht NA allerdings nicht für einen wirklich fehlenden Wert, sondern gibt uns echte Informationen. Die NAs *Included.in.systematic.analysis* geben uns zum Beispiel Aufschluss darüber, welche ATAC-peaks nicht in der späteren Analyse vom Research team weiterverwendet wurde.
Alle NAs in *TSS* markieren distal Enhancer und Gene, die in ihrer unmittelbaren Nähe (100kbs) kein Gen besitzen, werden ebenfalls mit NA gekennzeichnet. <p>
Daher müssen wir auch keine fehlenden Werte entfernen.

<h2> (p-Value Filtering)

In [6]:
Minimum_pvalue = ATAC_data['_-log10_bestPvalue'].min()
number_minimum = (ATAC_data['_-log10_bestPvalue'] == Minimum_pvalue).sum()
ATAC_data[ATAC_data['_-log10_bestPvalue'] == Minimum_pvalue]

,ImmGenATAC1219.peakID,chrom,Summit,mm10.60way.phastCons_scores,_-log10_bestPvalue,Included.in.systematic.analysis,TSS,genes.within.100Kb,LTHSC.34-.BM,LTHSC.34+.BM,...,DC.4+.Sp,DC.8+.Sp,DC.pDC.Sp,DC.103+11b+.SI,DC.103+11b-.SI,FRC.SLN,IAP.SLN,BEC.SLN,LEC.SLN,Ep.MEChi.Th
64295,ImmGenATAC1219.peak_64296,chr10,122421608,0.0,0.0,NaN,NaN,Avpr1a,0.41,0.1,...,0.1,0.10,0.11,0.11,0.11,0.11,0.34,0.11,0.11,0.11
508409,ImmGenATAC1219.peak_508410,chrX,106733598,0.0,0.0,NaN,NaN,NaN,0.41,0.1,...,0.1,0.29,0.11,0.47,0.52,0.14,0.10,0.11,0.51,0.11


**P-Value filtering**: <p>
Wir filtern die Werte nach P-Values und untersuchen den Datensatz nach auffällig niedrigen vs hohen p-Values. <br> Eventuell sollten hohe p-Values (niedriger log) aus dem Datensatz rausgenommen werden. <br> Eventuell könnte man eine bestimmten p-Value Grenze setzen (z.B. 0.05) und dann Werte darüber excluden. <p>
*TO BE CONTINUED...*

<h2> Filtering low-signal peaks or low-variance features
<h3> Filtering low-signal peaks

In [31]:
ATAC_pfiltered = ATAC_data[ATAC_data['_-log10_bestPvalue'] > -np.log10(0.05)]
ATAC_pfiltered
#len(ATAC_pfiltered)

,ImmGenATAC1219.peakID,chrom,Summit,mm10.60way.phastCons_scores,_-log10_bestPvalue,Included.in.systematic.analysis,TSS,genes.within.100Kb,LTHSC.34-.BM,LTHSC.34+.BM,...,DC.4+.Sp,DC.8+.Sp,DC.pDC.Sp,DC.103+11b+.SI,DC.103+11b-.SI,FRC.SLN,IAP.SLN,BEC.SLN,LEC.SLN,Ep.MEChi.Th
2,ImmGenATAC1219.peak_3,chr1,3120109,0.07,10.80,1.0,NaN,NaN,2.36,0.10,...,0.87,0.54,2.72,0.95,0.11,63.38,8.92,1.33,1.04,0.11
3,ImmGenATAC1219.peak_4,chr1,3121485,0.15,3.02,1.0,NaN,NaN,0.41,0.10,...,0.44,1.83,0.66,0.11,0.92,13.50,0.98,1.28,1.04,0.11
4,ImmGenATAC1219.peak_5,chr1,3372787,0.03,1.31,NaN,NaN,NaN,0.41,0.10,...,0.44,0.10,0.66,1.79,0.51,0.92,0.75,1.33,1.61,4.50
5,ImmGenATAC1219.peak_6,chr1,3399217,0.06,2.39,1.0,NaN,NaN,2.36,1.64,...,1.34,0.29,0.23,0.89,0.11,0.53,1.40,0.90,2.87,9.09
6,ImmGenATAC1219.peak_7,chr1,3400115,0.44,2.57,1.0,NaN,NaN,0.41,0.10,...,0.87,1.93,0.59,0.11,1.39,2.58,0.75,2.30,2.34,11.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512588,ImmGenATAC1219.peak_512589,chrY,90811728,0.00,2.33,1.0,NaN,Erdr1,0.41,7.41,...,4.98,4.47,2.83,4.93,4.92,5.13,9.13,2.21,6.53,6.11
512589,ImmGenATAC1219.peak_512590,chrY,90812084,0.00,3.12,1.0,NaN,Erdr1,2.36,8.79,...,4.03,4.07,5.87,5.36,3.99,7.12,3.57,2.64,5.59,3.64
512590,ImmGenATAC1219.peak_512591,chrY,90812450,0.00,3.99,1.0,NaN,Erdr1,4.37,8.79,...,3.81,3.34,4.27,6.73,5.53,7.21,5.96,5.17,6.53,6.11
512591,ImmGenATAC1219.peak_512592,chrY,90812906,0.00,3.21,1.0,NaN,Erdr1,0.41,7.41,...,4.28,5.55,4.15,6.88,7.16,6.21,8.75,6.83,8.14,4.64


**ATAC_data nur mit Werten, die einen p-Value kleiner 0.05 haben** <p>
1/3 von Werten haben einen p-Value der größer ist als 0.05, daher könnten diese Werte eine geringere statistische Signifikanz haben. Für unsere Nalyse ist es allerdings nicht sinnvoll, all diese Werte zu vernachlässigen.

In [ ]:
ATAC_NA_pvalue = ATAC_data[
    (ATAC_data['_-log10_bestPvalue'] < -np.log10(0.05)) &
    (ATAC_data['Included.in.systematic.analysis'].isnull())
]

#Zeilen, die in Included.in.systematic.analysis NA stehen haben und gleichzeitig einen pValue von über 0.05 haben
# --> 150265 Werte (das entspricht allen Werten, deren p-Value über 0.05 ist)
# daher ist davon auszugehen, dass all diese Werte von Yoshida und Co. entfernt wurden (+noch mehr) 

len(ATAC_NA_pvalue)

    

150265

**Pre-Filtering im Yoshida Paper** <p>
Alle Werte, die einen p-Value von größer als 0.05 haben, wurden im Yoshida Paper auch von der Analyse ausgeschlossen (150265). Darüber hinaus haben Yoshida et al. noch weitere Werte nach eigenen bestimmten Kriterien Filering-Prozessen entfernt. Es würde für unsere Analyse daher Sinn ergeben, diesen Threshold von 0.05 anzuwenden, da diese Werte mindestens auch im Research von Paper von Yoshida entfernt wurden.
